## Nearest Neighbor Class Fidelity (cf) Metric Implementation

In [ ]:
import numpy as np
from sklearn.neighbors import NearestNeighbors

In [ ]:
class NearestNeighborClassFidelity:
    def __init__(self, X: np.ndarray, y: np.array, k = 10):
        self.X = X
        self.y = y
        self.k = k

        self.knn: NearestNeighbors = NearestNeighbors(n_neighbors=k).fit(X)

    def get_nearest_neighbors(self, x, k):
        k = max(k, self.k)

        return self.knn.kneighbors(x, n_neighbors=k)
    
    def get_nearest_neighbors_of_class(self, x, y, k):
        
#




In [ ]:
import numpy as np
from sklearn.neighbors import NearestNeighbors

class NearestNeighborClassFidelity:
    def __init__(self, X: np.ndarray, y: np.ndarray, k=10):
        self.X = X          # punkty w przestrzeni niskowymiarowej
        self.y = y          # etykiety klas
        self.k = k          # liczba sąsiadów
        self.n_samples = X.shape[0]

        # Tworzymy model KNN
        self.knn = NearestNeighbors(n_neighbors=k+1).fit(X)  # +1, bo punkt sam w sobie
     
    def get_nearest_neighbors(self, x):
        distances, indices = self.knn.kneighbors(x, n_neighbors=self.k+1)
        # pomijamy punkt sam w sobie
        return indices[:, 1:], distances[:, 1:]

    def get_nn_counts(self):
        """Zwraca listę słowników {klasa: liczba sąsiadów} dla każdego punktu"""
        nn_counts = []
        indices, _ = self.get_nearest_neighbors(self.X)
        for n_idx, neigh in enumerate(indices):
            labels = self.y[neigh]
            unique, counts = np.unique(labels, return_counts=True)
            nn_counts.append(dict(zip(unique, counts)))
        return nn_counts

    def compute_cf(self):
        """Oblicza cf dla wszystkich punktów"""
        nn_counts = self.get_nn_counts()
        cf_list = []
        for n_idx, counts in enumerate(nn_counts):
            M = len(counts)  # liczba klas w sąsiedztwie
            k = sum(counts.values())  # liczba wszystkich sąsiadów
            # cf = średnia proporcja sąsiadów w każdej klasie
            cf = np.sum([v/k for v in counts.values()]) / M
            cf_list.append(cf)
        return np.array(cf_list)

    def compute_CF(self):
        """Oblicza całkowite CF dla zbioru"""
        cf_list = self.compute_cf()
        N_max = len(cf_list)  # normalizacja
        CF = np.sum(cf_list) / N_max
        return CF